<a href="https://colab.research.google.com/github/Anshuman22coder/BUG_FEEDBACK/blob/main/Merging_questions_14408_60.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations


# ============================================================
# CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# INPUT FILE
# ------------------------------------------------------------
# This should be the file containing the 14,408 submissions
# AFTER canonical-question mapping.
#
# Required columns:
# canonical_question_id
# canonical_question
# concept_type
# concept_name
# question
# code
# exec_feedback
# ai_explanation
# cluster
# ------------------------------------------------------------

input_file = "/content/master_60_questions_kmeans16.xlsx"

# Output file
output_file = "/content/k16_canonical_question_overlap_analysis.xlsx"

# Expected number of canonical questions
EXPECTED_CANONICAL_QUESTIONS = 60

# Expected number of clusters
EXPECTED_CLUSTERS = 16


# ============================================================
# 1. LOAD DATASET
# ============================================================

df = pd.read_excel(
    input_file,
    sheet_name="14408_Mapped_Data"
)

print("=" * 80)
print("K-MEANS CANONICAL QUESTION OVERLAP ANALYSIS")
print("=" * 80)

print("\nDataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 2. BASIC CHECKS
# ============================================================

required_columns = [
    "canonical_question_id",
    "canonical_question",
    "concept_type",
    "concept_name",
    "question",
    "code",
    "exec_feedback",
    "ai_explanation",
    "cluster"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


print("\nNumber of submissions:", len(df))

print(
    "Number of canonical questions:",
    df["canonical_question_id"].nunique()
)

print(
    "Number of clusters:",
    df["cluster"].nunique()
)


# ============================================================
# 3. CLEAN / STANDARDIZE CANONICAL QUESTION ID
# ============================================================

df["canonical_question_id"] = (
    df["canonical_question_id"]
    .astype(str)
    .str.strip()
)


# Remove accidental ".0" if IDs came from Excel
df["canonical_question_id"] = (
    df["canonical_question_id"]
    .str.replace(r"\.0$", "", regex=True)
)


# ============================================================
# 4. CLEAN CLUSTER COLUMN
# ============================================================

# Convert cluster to numeric
df["cluster"] = pd.to_numeric(
    df["cluster"],
    errors="coerce"
)

# Check invalid cluster values
invalid_cluster_rows = df["cluster"].isna().sum()

if invalid_cluster_rows > 0:
    print(
        f"\nWARNING: {invalid_cluster_rows} rows have "
        "invalid/missing cluster values."
    )

    print(
        "These rows will be excluded from cluster overlap analysis."
    )


# Keep only rows with valid cluster assignments
analysis_df = df[
    df["cluster"].notna()
].copy()

# Convert cluster to integer
analysis_df["cluster"] = (
    analysis_df["cluster"]
    .astype(int)
)


# ============================================================
# 5. CHECK CLUSTER RANGE
# ============================================================

expected_cluster_ids = set(
    range(EXPECTED_CLUSTERS)
)

actual_cluster_ids = set(
    analysis_df["cluster"].unique()
)

unexpected_clusters = (
    actual_cluster_ids - expected_cluster_ids
)

if unexpected_clusters:
    print(
        "\nWARNING: Unexpected cluster IDs found:",
        sorted(unexpected_clusters)
    )


print(
    "\nValid rows for cluster analysis:",
    len(analysis_df)
)


# ============================================================
# 6. CHECK CANONICAL QUESTION COUNT
# ============================================================

canonical_ids = (
    analysis_df["canonical_question_id"]
    .dropna()
    .unique()
)

print(
    "\nCanonical questions found:",
    len(canonical_ids)
)

if len(canonical_ids) != EXPECTED_CANONICAL_QUESTIONS:

    print(
        "\nWARNING:"
        f" Expected {EXPECTED_CANONICAL_QUESTIONS} "
        f"canonical questions but found {len(canonical_ids)}."
    )


# ============================================================
# 7. CREATE CANONICAL QUESTION MASTER TABLE
# ============================================================
#
# One row = one canonical question.
#
# We retain:
#
# canonical_question_id
# canonical_question
# concept_type
# concept_name
#
# ============================================================

canonical_master = (
    analysis_df[
        [
            "canonical_question_id",
            "canonical_question",
            "concept_type",
            "concept_name"
        ]
    ]
    .drop_duplicates(
        subset="canonical_question_id"
    )
    .copy()
)


# ============================================================
# 8. CHECK FOR MULTIPLE NAMES/DESCRIPTIONS FOR SAME ID
# ============================================================

canonical_consistency = (
    analysis_df
    .groupby("canonical_question_id")
    .agg(
        canonical_question_variants=(
            "canonical_question",
            "nunique"
        ),
        concept_type_variants=(
            "concept_type",
            "nunique"
        ),
        concept_name_variants=(
            "concept_name",
            "nunique"
        )
    )
    .reset_index()
)


inconsistent_canonical_questions = (
    canonical_consistency[
        (canonical_consistency["canonical_question_variants"] > 1)
        |
        (canonical_consistency["concept_type_variants"] > 1)
        |
        (canonical_consistency["concept_name_variants"] > 1)
    ]
)


if len(inconsistent_canonical_questions) > 0:

    print(
        "\nWARNING:"
        " Some canonical question IDs have inconsistent metadata."
    )

    display(
        inconsistent_canonical_questions
    )


# ============================================================
# 9. CLUSTER-WISE CANONICAL QUESTION COUNTS
# ============================================================
#
# This is now the MAIN dataset.
#
# Example:
#
# Cluster 0:
#
# Q01 -> 150 submissions
# Q07 -> 80 submissions
# Q12 -> 40 submissions
#
# Importantly:
#
# Q01 represents ALL raw variants that were mapped
# to canonical question Q01.
#
# ============================================================

cluster_question_counts = (
    analysis_df
    .groupby(
        [
            "cluster",
            "canonical_question_id",
            "canonical_question",
            "concept_type",
            "concept_name"
        ],
        as_index=False
    )
    .size()
    .rename(
        columns={
            "size": "submission_count"
        }
    )
)


# ============================================================
# 10. TOTAL SUBMISSIONS PER CLUSTER
# ============================================================

cluster_totals = (
    analysis_df
    .groupby("cluster")
    .size()
    .reset_index(
        name="cluster_total_submissions"
    )
)


# ============================================================
# 11. QUESTION PERCENTAGE WITHIN EACH CLUSTER
# ============================================================
#
# IMPORTANT:
#
# This percentage answers:
#
# "Of all submissions in Cluster X,
#  what percentage belong to canonical question Q?"
#
# Formula:
#
# question submissions in cluster
# -------------------------------- × 100
# total submissions in cluster
#
# ============================================================

cluster_question_counts = (
    cluster_question_counts
    .merge(
        cluster_totals,
        on="cluster",
        how="left"
    )
)


cluster_question_counts[
    "percentage_of_cluster"
] = np.where(
    cluster_question_counts[
        "cluster_total_submissions"
    ] > 0,

    (
        cluster_question_counts[
            "submission_count"
        ]
        /
        cluster_question_counts[
            "cluster_total_submissions"
        ]
        * 100
    ),

    0
)


cluster_question_counts[
    "percentage_of_cluster"
] = (
    cluster_question_counts[
        "percentage_of_cluster"
    ]
    .round(2)
)


# ============================================================
# 12. TOTAL SUBMISSIONS PER CANONICAL QUESTION
# ============================================================
#
# This is VERY IMPORTANT for your analysis.
#
# For Q01:
#
# Q01 Cluster 0 = 100
# Q01 Cluster 1 = 50
# Q01 Cluster 2 = 150
#
# Total Q01 = 300
#
# ============================================================

canonical_totals = (
    analysis_df
    .groupby("canonical_question_id")
    .size()
    .reset_index(
        name="canonical_question_total_submissions"
    )
)


# ============================================================
# 13. MERGE CANONICAL TOTALS
# ============================================================

cluster_question_counts = (
    cluster_question_counts
    .merge(
        canonical_totals,
        on="canonical_question_id",
        how="left"
    )
)


# ============================================================
# 14. CALCULATE PERCENTAGE WITHIN EACH CANONICAL QUESTION
# ============================================================
#
# THIS is the percentage you were asking about earlier.
#
# Example:
#
# Q01 has 1,000 submissions.
#
# Cluster 0 = 300
# Cluster 1 = 200
# Cluster 2 = 500
#
# Then:
#
# Q01 cluster 0 percentage = 300 / 1000 × 100
#                           = 30%
#
# Q01 cluster 1 percentage = 200 / 1000 × 100
#                           = 20%
#
# ============================================================

cluster_question_counts[
    "percentage_of_canonical_question"
] = np.where(

    cluster_question_counts[
        "canonical_question_total_submissions"
    ] > 0,

    (
        cluster_question_counts[
            "submission_count"
        ]
        /
        cluster_question_counts[
            "canonical_question_total_submissions"
        ]
        * 100
    ),

    0
)


cluster_question_counts[
    "percentage_of_canonical_question"
] = (
    cluster_question_counts[
        "percentage_of_canonical_question"
    ]
    .round(2)
)


# ============================================================
# 15. SORT CLUSTER QUESTIONS
# ============================================================

cluster_question_counts = (
    cluster_question_counts
    .sort_values(
        [
            "cluster",
            "submission_count"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 16. CREATE HUMAN-READABLE QUESTION LIST PER CLUSTER
# ============================================================

cluster_question_list = []


for cluster, group in (
    cluster_question_counts
    .groupby("cluster")
):

    questions = []

    for _, row in group.iterrows():

        questions.append(
            f"{row['canonical_question_id']}: "
            f"{row['canonical_question']} "
            f"({int(row['submission_count'])} submissions, "
            f"{row['percentage_of_cluster']:.2f}% of cluster)"
        )

    cluster_question_list.append({

        "cluster": cluster,

        "number_of_canonical_questions": len(group),

        "total_submissions": int(
            group["submission_count"].sum()
        ),

        "questions": "\n".join(
            questions
        )
    })


cluster_question_list = pd.DataFrame(
    cluster_question_list
)


# ============================================================
# 17. FIND CANONICAL QUESTION PAIRS WITHIN EACH CLUSTER
# ============================================================
#
# MAIN OVERLAP ANALYSIS
#
# If Cluster 0 contains:
#
# Q01
# Q02
# Q07
#
# We generate:
#
# Q01 <-> Q02
# Q01 <-> Q07
# Q02 <-> Q07
#
# ============================================================

pairwise_overlap = []


for cluster, group in (
    cluster_question_counts
    .groupby("cluster")
):

    questions = group.to_dict(
        "records"
    )

    for q1, q2 in combinations(
        questions,
        2
    ):

        pairwise_overlap.append({

            "cluster": cluster,

            "question_1_id":
                q1[
                    "canonical_question_id"
                ],

            "question_1":
                q1[
                    "canonical_question"
                ],

            "question_1_concept_type":
                q1[
                    "concept_type"
                ],

            "question_1_concept_name":
                q1[
                    "concept_name"
                ],

            "question_1_submissions":
                int(
                    q1[
                        "submission_count"
                    ]
                ),

            "question_1_percentage_of_cluster":
                q1[
                    "percentage_of_cluster"
                ],

            "question_1_percentage_of_own_submissions":
                q1[
                    "percentage_of_canonical_question"
                ],

            "question_2_id":
                q2[
                    "canonical_question_id"
                ],

            "question_2":
                q2[
                    "canonical_question"
                ],

            "question_2_concept_type":
                q2[
                    "concept_type"
                ],

            "question_2_concept_name":
                q2[
                    "concept_name"
                ],

            "question_2_submissions":
                int(
                    q2[
                        "submission_count"
                    ]
                ),

            "question_2_percentage_of_cluster":
                q2[
                    "percentage_of_cluster"
                ],

            "question_2_percentage_of_own_submissions":
                q2[
                    "percentage_of_canonical_question"
                ],

            "co_occurrence": True,

            "combined_submissions":
                (
                    int(
                        q1[
                            "submission_count"
                        ]
                    )
                    +
                    int(
                        q2[
                            "submission_count"
                        ]
                    )
                )
        })


pairwise_overlap_df = pd.DataFrame(
    pairwise_overlap
)


# ============================================================
# 18. CLUSTER-LEVEL SUMMARY
# ============================================================

cluster_summary = []


for cluster, group in (
    cluster_question_counts
    .groupby("cluster")
):

    n_questions = len(group)

    n_pairs = (
        n_questions
        * (n_questions - 1)
        // 2
    )

    total_submissions = (
        group[
            "submission_count"
        ].sum()
    )

    # Group already sorted by submission_count
    dominant_row = (
        group.iloc[0]
    )

    dominant_percentage = (
        dominant_row[
            "percentage_of_cluster"
        ]
    )

    cluster_summary.append({

        "cluster": cluster,

        "number_of_canonical_questions":
            n_questions,

        "number_of_question_pairs":
            n_pairs,

        "total_submissions":
            int(
                total_submissions
            ),

        "dominant_question_id":
            dominant_row[
                "canonical_question_id"
            ],

        "dominant_question":
            dominant_row[
                "canonical_question"
            ],

        "dominant_question_concept":
            dominant_row[
                "concept_name"
            ],

        "dominant_question_submissions":
            int(
                dominant_row[
                    "submission_count"
                ]
            ),

        "dominant_question_percentage":
            dominant_percentage
    })


cluster_summary_df = pd.DataFrame(
    cluster_summary
)


# ============================================================
# 19. CLASSIFY CLUSTER COMPOSITION
# ============================================================

def classify_cluster(row):

    n = row[
        "number_of_canonical_questions"
    ]

    dominant = row[
        "dominant_question_percentage"
    ]

    if n == 1:

        return "Single-canonical-question cluster"

    elif dominant >= 80:

        return "Highly dominated by one canonical question"

    elif dominant >= 60:

        return "Mostly dominated by one canonical question"

    else:

        return "Mixed-canonical-question cluster"


cluster_summary_df[
    "cluster_composition"
] = (
    cluster_summary_df
    .apply(
        classify_cluster,
        axis=1
    )
)


# ============================================================
# 20. CANONICAL QUESTION × CLUSTER COUNT MATRIX
# ============================================================
#
# Rows    = Q01–Q60
# Columns = Cluster 0–15
# Values  = submission count
#
# ============================================================

question_cluster_counts = pd.pivot_table(

    analysis_df,

    index=[
        "canonical_question_id"
    ],

    columns="cluster",

    values="question",

    aggfunc="count",

    fill_value=0
)


question_cluster_counts = (
    question_cluster_counts
    .reset_index()
)


# ============================================================
# 21. ENSURE ALL 16 CLUSTERS ARE PRESENT
# ============================================================

for cluster_id in range(
    EXPECTED_CLUSTERS
):

    if cluster_id not in (
        question_cluster_counts.columns
    ):

        question_cluster_counts[
            cluster_id
        ] = 0


# Sort cluster columns
cluster_columns_sorted = [
    c
    for c in range(
        EXPECTED_CLUSTERS
    )
]


question_cluster_counts = (
    question_cluster_counts[
        ["canonical_question_id"]
        +
        cluster_columns_sorted
    ]
)


# ============================================================
# 22. ADD CANONICAL QUESTION METADATA
# ============================================================

question_cluster_counts = (
    question_cluster_counts
    .merge(
        canonical_master,
        on="canonical_question_id",
        how="left"
    )
)


# Put metadata first
question_cluster_counts = (
    question_cluster_counts[
        [
            "canonical_question_id",
            "canonical_question",
            "concept_type",
            "concept_name"
        ]
        +
        cluster_columns_sorted
    ]
)


# ============================================================
# 23. CANONICAL QUESTION × CLUSTER PERCENTAGE MATRIX
# ============================================================
#
# IMPORTANT:
#
# Each ROW sums approximately to 100%.
#
# Example:
#
# Q01:
#
# Cluster 0 = 30%
# Cluster 1 = 20%
# Cluster 2 = 50%
#
# Total = 100%
#
# ============================================================

question_cluster_percentage = (
    question_cluster_counts[
        [
            "canonical_question_id",
            "canonical_question",
            "concept_type",
            "concept_name"
        ]
        +
        cluster_columns_sorted
    ]
    .copy()
)


for cluster_id in cluster_columns_sorted:

    question_cluster_percentage[
        cluster_id
    ] = np.where(

        question_cluster_percentage[
            cluster_columns_sorted
        ].sum(axis=1) > 0,

        question_cluster_percentage[
            cluster_id
        ]
        /
        question_cluster_percentage[
            cluster_columns_sorted
        ].sum(axis=1)
        * 100,

        0
    )


question_cluster_percentage[
    cluster_columns_sorted
] = (
    question_cluster_percentage[
        cluster_columns_sorted
    ]
    .round(2)
)


# Rename cluster columns for clarity
question_cluster_percentage = (
    question_cluster_percentage
    .rename(
        columns={
            c: f"cluster_{c}_percentage"
            for c in cluster_columns_sorted
        }
    )
)


# ============================================================
# 24. CANONICAL QUESTION × CLUSTER BINARY MATRIX
# ============================================================
#
# 1 = canonical question occurs in cluster
# 0 = canonical question does not occur
#
# ============================================================

question_cluster_binary = (
    question_cluster_counts
    .copy()
)


question_cluster_binary[
    cluster_columns_sorted
] = (
    question_cluster_binary[
        cluster_columns_sorted
    ]
    .apply(
        lambda col: (
            col > 0
        ).astype(int)
    )
)


# Rename clusters
question_cluster_binary = (
    question_cluster_binary
    .rename(
        columns={
            c: f"cluster_{c}"
            for c in cluster_columns_sorted
        }
    )
)


# ============================================================
# 25. HOW MANY CLUSTERS DOES EACH CANONICAL QUESTION APPEAR IN?
# ============================================================

question_cluster_summary = []


for _, row in (
    question_cluster_binary
    .iterrows()
):

    present_clusters = []

    for cluster_id in cluster_columns_sorted:

        column_name = (
            f"cluster_{cluster_id}"
        )

        if row[
            column_name
        ] == 1:

            present_clusters.append(
                cluster_id
            )


    question_cluster_summary.append({

        "canonical_question_id":
            row[
                "canonical_question_id"
            ],

        "canonical_question":
            row[
                "canonical_question"
            ],

        "concept_type":
            row[
                "concept_type"
            ],

        "concept_name":
            row[
                "concept_name"
            ],

        "number_of_clusters":
            len(
                present_clusters
            ),

        "clusters":
            ", ".join(
                map(
                    str,
                    present_clusters
                )
            )
    })


question_cluster_summary_df = (
    pd.DataFrame(
        question_cluster_summary
    )
)


# ============================================================
# 26. FIND CANONICAL QUESTIONS APPEARING IN MULTIPLE CLUSTERS
# ============================================================

multi_cluster_questions = (

    question_cluster_summary_df[
        question_cluster_summary_df[
            "number_of_clusters"
        ] > 1
    ]

    .sort_values(
        "number_of_clusters",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


# ============================================================
# 27. QUESTION-LEVEL CLUSTER CONCENTRATION
# ============================================================
#
# For each canonical question:
#
# cluster_concentration =
#
# largest cluster count
# --------------------- × 100
# total submissions of that canonical question
#
# Example:
#
# Q01 = 1000 submissions
# Cluster 0 = 700
#
# concentration = 700/1000 × 100
#                = 70%
#
# ============================================================

question_distribution_summary = []


for _, row in (
    question_cluster_counts
    .iterrows()
):

    counts = [
        row[c]
        for c in cluster_columns_sorted
    ]

    total = sum(counts)

    if total > 0:

        dominant_count = max(
            counts
        )

        dominant_cluster = (
            cluster_columns_sorted[
                np.argmax(counts)
            ]
        )

        concentration = (
            dominant_count
            /
            total
            *
            100
        )

    else:

        dominant_count = 0

        dominant_cluster = None

        concentration = 0


    question_distribution_summary.append({

        "canonical_question_id":
            row[
                "canonical_question_id"
            ],

        "canonical_question":
            row[
                "canonical_question"
            ],

        "concept_type":
            row[
                "concept_type"
            ],

        "concept_name":
            row[
                "concept_name"
            ],

        "total_submissions":
            int(total),

        "dominant_cluster":
            dominant_cluster,

        "dominant_cluster_count":
            int(
                dominant_count
            ),

        "cluster_concentration":
            round(
                concentration,
                2
            )
    })


question_distribution_summary_df = (
    pd.DataFrame(
        question_distribution_summary
    )
)


# ============================================================
# 28. CLUSTERS ABOVE 1%, 5%, 10%
# ============================================================
#
# These percentages are WITHIN EACH CANONICAL QUESTION.
#
# Example:
#
# Q01:
#
# C0 = 60%
# C1 = 20%
# C2 = 10%
# C3 = 5%
# C4 = 5%
#
# clusters_above_10pct = 3
#
# C0, C1 and C2
#
# ============================================================

threshold_summary = []


for _, row in (
    question_cluster_percentage
    .iterrows()
):

    percentages = [
        row[
            f"cluster_{c}_percentage"
        ]
        for c in cluster_columns_sorted
    ]


    threshold_summary.append({

        "canonical_question_id":
            row[
                "canonical_question_id"
            ],

        "clusters_above_1pct":
            sum(
                p > 1
                for p in percentages
            ),

        "clusters_above_5pct":
            sum(
                p > 5
                for p in percentages
            ),

        "clusters_above_10pct":
            sum(
                p > 10
                for p in percentages
            )
    })


threshold_summary_df = (
    pd.DataFrame(
        threshold_summary
    )
)


# ============================================================
# 29. MERGE QUESTION DISTRIBUTION SUMMARY
# ============================================================

question_distribution_summary_df = (
    question_distribution_summary_df
    .merge(
        threshold_summary_df,
        on="canonical_question_id",
        how="left"
    )
)


# ============================================================
# 30. CLUSTER OVERLAP MATRIX
# ============================================================
#
# This matrix answers:
#
# How many canonical questions occur
# in BOTH Cluster A and Cluster B?
#
# Example:
#
#             C0  C1  C2
# C0          10   4   7
# C1           4   12  5
# C2           7   5   15
#
# C0-C1 overlap = 4 canonical questions
#
# ============================================================

binary_for_overlap = (
    question_cluster_binary
    .set_index(
        "canonical_question_id"
    )[
        [
            f"cluster_{c}"
            for c in cluster_columns_sorted
        ]
    ]
)


cluster_overlap_matrix = (
    binary_for_overlap.T
    .dot(
        binary_for_overlap
    )
)


# Rename rows/columns
cluster_overlap_matrix.index = [
    f"Cluster {c}"
    for c in cluster_columns_sorted
]

cluster_overlap_matrix.columns = [
    f"Cluster {c}"
    for c in cluster_columns_sorted
]


cluster_overlap_matrix = (
    cluster_overlap_matrix
    .reset_index()
    .rename(
        columns={
            "index": "cluster"
        }
    )
)


# ============================================================
# 31. PAIRWISE CANONICAL QUESTION OVERLAP ACROSS CLUSTERS
# ============================================================
#
# This is another important overlap measure.
#
# For Q01 and Q07:
#
# Q01 appears in C0,C1,C2
# Q07 appears in C0,C2,C5
#
# Common clusters:
#
# C0,C2
#
# Therefore:
#
# number_of_common_clusters = 2
#
# ============================================================

question_pair_overlap = []


# Binary matrix indexed by canonical question
binary_matrix = (
    question_cluster_binary
    .set_index(
        "canonical_question_id"
    )
)


question_ids = (
    binary_matrix.index.tolist()
)


# Metadata lookup
metadata_lookup = (
    canonical_master
    .set_index(
        "canonical_question_id"
    )
)


for q1, q2 in combinations(
    question_ids,
    2
):

    common_clusters = []

    q1_clusters = []
    q2_clusters = []


    for cluster_id in cluster_columns_sorted:

        column_name = (
            f"cluster_{cluster_id}"
        )

        q1_present = (
            binary_matrix.loc[
                q1,
                column_name
            ] == 1
        )

        q2_present = (
            binary_matrix.loc[
                q2,
                column_name
            ] == 1
        )


        if q1_present:

            q1_clusters.append(
                cluster_id
            )

        if q2_present:

            q2_clusters.append(
                cluster_id
            )

        if (
            q1_present
            and
            q2_present
        ):

            common_clusters.append(
                cluster_id
            )


    # Jaccard similarity
    union_clusters = sorted(
        set(q1_clusters)
        |
        set(q2_clusters)
    )

    if len(union_clusters) > 0:

        jaccard_similarity = (
            len(common_clusters)
            /
            len(union_clusters)
        )

    else:

        jaccard_similarity = 0


    question_pair_overlap.append({

        "question_1_id":
            q1,

        "question_1":
            metadata_lookup.loc[
                q1,
                "canonical_question"
            ],

        "question_1_concept":
            metadata_lookup.loc[
                q1,
                "concept_name"
            ],

        "question_1_number_of_clusters":
            len(q1_clusters),

        "question_1_clusters":
            ", ".join(
                map(
                    str,
                    q1_clusters
                )
            ),

        "question_2_id":
            q2,

        "question_2":
            metadata_lookup.loc[
                q2,
                "canonical_question"
            ],

        "question_2_concept":
            metadata_lookup.loc[
                q2,
                "concept_name"
            ],

        "question_2_number_of_clusters":
            len(q2_clusters),

        "question_2_clusters":
            ", ".join(
                map(
                    str,
                    q2_clusters
                )
            ),

        "number_of_common_clusters":
            len(
                common_clusters
            ),

        "common_clusters":
            ", ".join(
                map(
                    str,
                    common_clusters
                )
            ),

        "jaccard_cluster_similarity":
            round(
                jaccard_similarity,
                4
            )
    })


question_pair_overlap_df = (
    pd.DataFrame(
        question_pair_overlap
    )
)


# ============================================================
# 32. SORT PAIRWISE OVERLAP
# ============================================================

question_pair_overlap_df = (
    question_pair_overlap_df
    .sort_values(
        [
            "number_of_common_clusters",
            "jaccard_cluster_similarity"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 33. CONCEPT × CLUSTER COUNT
# ============================================================
#
# Since you have:
#
# concept_type
# concept_name
#
# we can also see how the six programming concepts
# are distributed across the clusters.
#
# ============================================================

concept_cluster_counts = (
    analysis_df
    .groupby(
        [
            "concept_type",
            "concept_name",
            "cluster"
        ],
        as_index=False
    )
    .size()
    .rename(
        columns={
            "size": "submission_count"
        }
    )
)


# ============================================================
# 34. CONCEPT × CLUSTER MATRIX
# ============================================================

concept_cluster_matrix = pd.pivot_table(

    analysis_df,

    index=[
        "concept_type",
        "concept_name"
    ],

    columns="cluster",

    values="question",

    aggfunc="count",

    fill_value=0
)


concept_cluster_matrix = (
    concept_cluster_matrix
    .reset_index()
)


# Ensure all 16 clusters
for cluster_id in cluster_columns_sorted:

    if cluster_id not in (
        concept_cluster_matrix.columns
    ):

        concept_cluster_matrix[
            cluster_id
        ] = 0


concept_cluster_matrix = (
    concept_cluster_matrix[
        [
            "concept_type",
            "concept_name"
        ]
        +
        cluster_columns_sorted
    ]
)


# ============================================================
# 35. CONCEPT × CLUSTER PERCENTAGE
# ============================================================
#
# For each concept:
#
# percentage of its submissions in each cluster.
#
# ============================================================

concept_cluster_percentage = (
    concept_cluster_matrix
    .copy()
)


for _, row in concept_cluster_matrix.iterrows():

    concept_total = sum(
        row[c]
        for c in cluster_columns_sorted
    )

    mask = (
        concept_cluster_percentage[
            "concept_type"
        ]
        ==
        row[
            "concept_type"
        ]
    ) & (
        concept_cluster_percentage[
            "concept_name"
        ]
        ==
        row[
            "concept_name"
        ]
    )

    for c in cluster_columns_sorted:

        if concept_total > 0:

            concept_cluster_percentage.loc[
                mask,
                c
            ] = (
                row[c]
                /
                concept_total
                *
                100
            )

        else:

            concept_cluster_percentage.loc[
                mask,
                c
            ] = 0


concept_cluster_percentage[
    cluster_columns_sorted
] = (
    concept_cluster_percentage[
        cluster_columns_sorted
    ]
    .round(2)
)


concept_cluster_percentage = (
    concept_cluster_percentage
    .rename(
        columns={
            c: f"cluster_{c}_percentage"
            for c in cluster_columns_sorted
        }
    )
)


# ============================================================
# 36. CONCEPT OVERLAP WITHIN CLUSTERS
# ============================================================
#
# This tells us which concepts coexist inside each cluster.
#
# Example:
#
# Cluster 0:
#
# Conditionals
# Loops
# Strings
#
# Therefore these concepts overlap in Cluster 0.
#
# ============================================================

concept_cluster_binary = (
    concept_cluster_counts
    .copy()
)


concept_cluster_binary[
    "present"
] = 1


concept_cluster_binary_matrix = (
    pd.pivot_table(

        concept_cluster_binary,

        index=[
            "concept_type",
            "concept_name"
        ],

        columns="cluster",

        values="present",

        aggfunc="max",

        fill_value=0
    )
    .reset_index()
)


for cluster_id in cluster_columns_sorted:

    if cluster_id not in (
        concept_cluster_binary_matrix.columns
    ):

        concept_cluster_binary_matrix[
            cluster_id
        ] = 0


concept_cluster_binary_matrix = (
    concept_cluster_binary_matrix[
        [
            "concept_type",
            "concept_name"
        ]
        +
        cluster_columns_sorted
    ]
)


# ============================================================
# 37. CLUSTER → CANONICAL QUESTION OVERLAP SUMMARY
# ============================================================

cluster_overlap_summary = []


for cluster, group in (
    cluster_question_counts
    .groupby("cluster")
):

    canonical_questions = (
        group[
            "canonical_question_id"
        ]
        .tolist()
    )

    concepts = (
        group[
            "concept_name"
        ]
        .dropna()
        .unique()
        .tolist()
    )


    cluster_overlap_summary.append({

        "cluster":
            cluster,

        "number_of_canonical_questions":
            len(
                canonical_questions
            ),

        "canonical_questions":
            ", ".join(
                canonical_questions
            ),

        "number_of_concepts":
            len(
                concepts
            ),

        "concepts":
            ", ".join(
                concepts
            )
    })


cluster_overlap_summary_df = (
    pd.DataFrame(
        cluster_overlap_summary
    )
)


# ============================================================
# 38. FINAL MASTER TABLE
# ============================================================
#
# One row = one of the 60 canonical questions.
#
# This is probably the MOST useful sheet for your report.
#
# ============================================================

master_analysis_df = (
    canonical_master
    .merge(
        canonical_totals,
        on="canonical_question_id",
        how="left"
    )
    .merge(
        question_distribution_summary_df[
            [
                "canonical_question_id",
                "dominant_cluster",
                "dominant_cluster_count",
                "cluster_concentration",
                "clusters_above_1pct",
                "clusters_above_5pct",
                "clusters_above_10pct"
            ]
        ],
        on="canonical_question_id",
        how="left"
    )
    .merge(
        question_cluster_summary_df[
            [
                "canonical_question_id",
                "number_of_clusters",
                "clusters"
            ]
        ],
        on="canonical_question_id",
        how="left"
    )
)


# ============================================================
# 39. ADD CLUSTER PERCENTAGES TO MASTER TABLE
# ============================================================

master_analysis_df = (
    master_analysis_df
    .merge(
        question_cluster_percentage,
        on=[
            "canonical_question_id",
            "canonical_question",
            "concept_type",
            "concept_name"
        ],
        how="left"
    )
)


# ============================================================
# 40. SORT MASTER TABLE BY CANONICAL QUESTION ID
# ============================================================

def canonical_sort_key(value):

    try:

        digits = (
            ''.join(
                filter(
                    str.isdigit,
                    str(value)
                )
            )
        )

        return int(digits)

    except:

        return 999999


master_analysis_df[
    "_sort_key"
] = (
    master_analysis_df[
        "canonical_question_id"
    ]
    .apply(
        canonical_sort_key
    )
)


master_analysis_df = (
    master_analysis_df
    .sort_values(
        "_sort_key"
    )
    .drop(
        columns="_sort_key"
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 41. CREATE CLUSTER PERCENTAGE COLUMNS IN MASTER
# ============================================================
#
# The master table now contains:
#
# cluster_0_percentage
# cluster_1_percentage
# ...
# cluster_15_percentage
#
# These are percentages OF THAT CANONICAL QUESTION.
#
# ============================================================


# ============================================================
# 42. VERIFY EACH QUESTION'S PERCENTAGES
# ============================================================

percentage_columns = [
    f"cluster_{c}_percentage"
    for c in cluster_columns_sorted
]


master_analysis_df[
    "percentage_sum_check"
] = (
    master_analysis_df[
        percentage_columns
    ]
    .sum(
        axis=1
    )
    .round(2)
)


# ============================================================
# 43. IDENTIFY QUESTIONS WITH DISTRIBUTED CLUSTERS
# ============================================================

highly_distributed_questions = (
    master_analysis_df[
        master_analysis_df[
            "number_of_clusters"
        ] >= 4
    ]
    .sort_values(
        [
            "number_of_clusters",
            "cluster_concentration"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 44. IDENTIFY QUESTIONS STRONGLY CONCENTRATED
# ============================================================

highly_concentrated_questions = (
    master_analysis_df[
        master_analysis_df[
            "cluster_concentration"
        ] >= 80
    ]
    .sort_values(
        "cluster_concentration",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 45. PRINT RESULTS
# ============================================================

print("\n")
print("=" * 80)
print("FINAL RESULTS")
print("=" * 80)


print(
    "\nTotal submissions:",
    len(df)
)


print(
    "Valid rows used:",
    len(analysis_df)
)


print(
    "Canonical questions:",
    analysis_df[
        "canonical_question_id"
    ].nunique()
)


print(
    "Clusters:",
    analysis_df[
        "cluster"
    ].nunique()
)


print("\n--- Cluster Summary ---")

display(
    cluster_summary_df
    .sort_values("cluster")
)


print("\n--- Canonical Questions Appearing in Multiple Clusters ---")

display(
    multi_cluster_questions
)


print("\n--- Canonical Question × Cluster Counts ---")

display(
    question_cluster_counts.head(60)
)


print("\n--- Canonical Question × Cluster Percentages ---")

display(
    question_cluster_percentage.head(60)
)


print("\n--- Pairwise Canonical Question Overlap ---")

if len(question_pair_overlap_df) > 0:

    display(
        question_pair_overlap_df.head(50)
    )

else:

    print(
        "No overlapping canonical question pairs found."
    )


print("\n--- Highly Distributed Canonical Questions ---")

display(
    highly_distributed_questions
)


print("\n--- Highly Concentrated Canonical Questions ---")

display(
    highly_concentrated_questions
)


print("\n--- Master Analysis ---")

display(
    master_analysis_df.head(60)
)


# ============================================================
# 46. SAVE EVERYTHING TO EXCEL
# ============================================================

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:


    # --------------------------------------------------------
    # Sheet 1
    # --------------------------------------------------------
    cluster_question_counts.to_excel(
        writer,
        sheet_name="Cluster_Questions",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 2
    # --------------------------------------------------------
    pairwise_overlap_df.to_excel(
        writer,
        sheet_name="Question_Pairs",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 3
    # --------------------------------------------------------
    cluster_summary_df.to_excel(
        writer,
        sheet_name="Cluster_Summary",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 4
    # --------------------------------------------------------
    cluster_question_list.to_excel(
        writer,
        sheet_name="Cluster_Question_List",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 5
    # --------------------------------------------------------
    question_cluster_counts.to_excel(
        writer,
        sheet_name="Question_Cluster_Counts",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 6
    # --------------------------------------------------------
    question_cluster_percentage.to_excel(
        writer,
        sheet_name="Question_Cluster_Percentage",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 7
    # --------------------------------------------------------
    question_cluster_binary.to_excel(
        writer,
        sheet_name="Question_Cluster_Binary",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 8
    # --------------------------------------------------------
    question_cluster_summary_df.to_excel(
        writer,
        sheet_name="Question_Cluster_Summary",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 9
    # --------------------------------------------------------
    multi_cluster_questions.to_excel(
        writer,
        sheet_name="Multi_Cluster_Questions",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 10
    # --------------------------------------------------------
    question_distribution_summary_df.to_excel(
        writer,
        sheet_name="Question_Distribution",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 11
    # --------------------------------------------------------
    cluster_overlap_matrix.to_excel(
        writer,
        sheet_name="Cluster_Overlap_Matrix",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 12
    # --------------------------------------------------------
    question_pair_overlap_df.to_excel(
        writer,
        sheet_name="Canonical_Pair_Overlap",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 13
    # --------------------------------------------------------
    concept_cluster_counts.to_excel(
        writer,
        sheet_name="Concept_Cluster_Counts",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 14
    # --------------------------------------------------------
    concept_cluster_matrix.to_excel(
        writer,
        sheet_name="Concept_Cluster_Matrix",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 15
    # --------------------------------------------------------
    concept_cluster_percentage.to_excel(
        writer,
        sheet_name="Concept_Cluster_Percentage",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 16
    # --------------------------------------------------------
    concept_cluster_binary_matrix.to_excel(
        writer,
        sheet_name="Concept_Cluster_Binary",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 17
    # --------------------------------------------------------
    cluster_overlap_summary_df.to_excel(
        writer,
        sheet_name="Cluster_Overlap_Summary",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 18
    # --------------------------------------------------------
    master_analysis_df.to_excel(
        writer,
        sheet_name="Master_60_Questions",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 19
    # --------------------------------------------------------
    highly_distributed_questions.to_excel(
        writer,
        sheet_name="Highly_Distributed",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 20
    # --------------------------------------------------------
    highly_concentrated_questions.to_excel(
        writer,
        sheet_name="Highly_Concentrated",
        index=False
    )


    # --------------------------------------------------------
    # Sheet 21
    # --------------------------------------------------------
    analysis_df.to_excel(
        writer,
        sheet_name="Mapped_14408_Data",
        index=False
    )


# ============================================================
# 47. FORMAT EXCEL FILE
# ============================================================

from openpyxl import load_workbook
from openpyxl.utils import get_column_letter


wb = load_workbook(
    output_file
)


for ws in wb.worksheets:

    # Freeze header row
    ws.freeze_panes = "A2"

    # Enable filter
    if ws.max_row > 1:
        ws.auto_filter.ref = (
            ws.dimensions
        )

    # Adjust column widths
    for column_cells in ws.columns:

        max_length = 0

        column_letter = (
            get_column_letter(
                column_cells[0].column
            )
        )

        for cell in column_cells:

            try:

                cell_length = len(
                    str(
                        cell.value
                    )
                )

                if cell_length > max_length:
                    max_length = cell_length

            except:

                pass


        # Keep Excel columns reasonable
        adjusted_width = min(
            max(
                max_length + 2,
                10
            ),
            60
        )

        ws.column_dimensions[
            column_letter
        ].width = adjusted_width


# Save formatted workbook
wb.save(
    output_file
)


# ============================================================
# 48. FINAL MESSAGE
# ============================================================

print("\n")
print("=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)

print(
    "\nOutput file:"
)

print(
    output_file
)

print("\n")
print("IMPORTANT INTERPRETATION:")
print(
    "cluster_X_percentage in Master_60_Questions "
    "means:"
)

print(
    "Number of submissions of that canonical question "
    "in Cluster X / "
    "total submissions of that canonical question × 100"
)

print("\nFor example:")

print(
    "Q01 Cluster 0 percentage = "
    "Q01 submissions in Cluster 0 / "
    "all Q01 submissions × 100"
)